# Load all Silver Delta data

In [0]:
from pyspark.sql.functions import *

# customers
customers_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customers/"

df_customers = (
    spark.read
    .format("delta")
    .load(customers_path)
)

#Orders
orders_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/orders/"

df_orders = (
    spark.read
    .format("delta")
    .load(orders_path)
)

# Products
products_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/products/"

df_products = (
    spark.read
    .format("delta")
    .load(products_path)
)

# Stores
stores_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/stores/"

df_stores = (
    spark.read
    .format("delta")
    .load(stores_path)
)

#Feedback

feedback_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customer_feedback/"

df_feedback = (
    spark.read
    .format("delta")
    .load(feedback_path)
)

#Activity
activity_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customer_activity/"

df_activity = (
    spark.read
    .format("delta")
    .load(activity_path)
)

# Create customer sales summary

In [0]:
customer_sales = (
    df_orders
    .groupBy("customer_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity"),
        sum("total_amount").alias("total_spend"),
        avg("total_amount").alias("average_order_value"),
        max("order_date").alias("last_order_date")
    )
)
display(customer_sales)

# Calculate feedback information

In [0]:
customer_feedback = (
    df_feedback
    .groupBy("customer_id")
    .agg(
        count("feedback_id").alias("total_feedback"),
        avg("rating").alias("average_rating")
    )
)

# Calculate activity information

In [0]:
customer_activity = (
    df_activity
    .groupBy("customer_id")
    .agg(
        count("*").alias("total_activities"),
        sum(
            when(col("activity_type") == "Product_View", 1).otherwise(0)
        ).alias("total_product_views"),
        sum(
            when(col("activity_type") == "Search", 1).otherwise(0)
        ).alias("total_searches"),
        sum(
            when(col("activity_type") == "Add_to_Cart", 1).otherwise(0)
        ).alias("total_add_to_cart"),
        sum(
            when(col("activity_type") == "Wishlist", 1).otherwise(0)
        ).alias("total_wishlist"),
        sum(
            when(col("activity_type") == "Purchase", 1).otherwise(0)
        ).alias("total_purchases")
    )
)

# Join everything to Customers

In [0]:
customer_360 = (
    df_customers
    .join(
        customer_sales,
        on="customer_id",
        how="left"
    )
    .join(
        customer_feedback,
        on="customer_id",
        how="left"
    )
    .join(
        customer_activity,
        on="customer_id",
        how="left"
    )
)

# Handle missing values

In [0]:
customer_360 = (
    customer_360
    .fillna({
        "total_orders": 0,
        "total_quantity": 0,
        "total_spend": 0,
        "average_order_value": 0,
        "total_feedback": 0,
        "average_rating": 0,
        "total_activities": 0,
        "total_product_views": 0,
        "total_searches": 0,
        "total_add_to_cart": 0,
        "total_wishlist": 0,
        "total_purchases": 0
    })
)

# Select the final Customer 360 columns

In [0]:
customer_360 = customer_360.select(
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "phone",
    "city",
    "state",
    "registration_date",
    "total_orders",
    "total_quantity",
    "total_spend",
    "average_order_value",
    "last_order_date",
    "total_feedback",
    "average_rating",
    "total_activities",
    "total_product_views",
    "total_searches",
    "total_add_to_cart",
    "total_wishlist",
    "total_purchases"
)

In [0]:
display(customer_360)

#Write Customer 360 to Gold

In [0]:

customer_360_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/gold/customer_360/"

(
    customer_360.write
    .format("delta")
    .mode("overwrite")
    .save(customer_360_path)
)

# Verfiy

In [0]:
df_customer_360 = (
    spark.read
    .format("delta")
    .load(customer_360_path)
)

display(df_customer_360)

print("Customer 360 records:", df_customer_360.count())